In [1]:
# ══════════════════════════════════════════════════════════════════
# NOTEBOOK 3 : INDICATOR SCORING
# Follows  exact thresholds from reference model
# Converts Prophet deviation/ratio back to raw values first
#
# thresholds:
#   STU:    25 / 20 / 15 / 0      (raw %)
#   Demand: 10 / 12 / 14 / 16     (ratio-based, see below)
#   PPI:     3 /  0 / -3 / -6     (raw deviation)
#   KSA:    15 / 25 / 50          (raw %)
#   Policy: 10 / 40 / 70 / 100    (raw score)
#   BDI:    30 / 60 / 90          (0-100 normalised)
#
# Output tables:
#   srm.prophet_indicator_scores
#   srm.prophet_policy_scores
# ══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

COMMODITIES   = ["Wheat","Corn","Rice","Soybean"]
CURRENT_MONTH = pd.Timestamp("2026-06-01")
FORECAST_MONTHS = pd.to_datetime(
    ["2026-07-01","2026-08-01","2026-09-01"]
)

STRING_COLS = [
    "ksa_top1_country","ksa_top1_country_lag1",
    "ksa_top3_countries","top_buyer_country",
    "individually_tracked_countries"
]

WEIGHTS = {
    "stu_deviation":         0.18,   # was 0.20
    "demand_ratio":          0.12,   # was 0.13
    "ppi_deviation":         0.11,   # was 0.12
    "ksa_deviation":         0.13,   # was 0.15
    "policy_risk_weighted":  0.18,   # was 0.20
    "bdi_ratio":             0.18,   # was 0.20
    "domestic_reserves_gap": 0.10    # NEW
}

INDICATORS_ORDERED = [
    "stu_deviation","demand_ratio","ppi_deviation",
    "ksa_deviation","policy_risk_weighted","bdi_ratio",
    "domestic_reserves_gap"   # NEW
]

INDICATOR_LABELS = {
    "stu_deviation":         "Global Stock-to-Use Ratio",
    "demand_ratio":          "Import Demand Pressure",
    "ppi_deviation":         "Production Potential Index",
    "ksa_deviation":         "KSA Import Concentration",
    "policy_risk_weighted":  "Export Restriction Status",
    "bdi_ratio":             "BDI Freight Index",
    "domestic_reserves_gap": "Domestic Strategic Reserves"   # NEW
}

def save_to_lakehouse(df_pandas, table_name, schema="srm"):
    full_name = f"{schema}.{table_name}"
    # Clean column names
    df_clean = df_pandas.copy()
    df_clean.columns = [
        c.replace("%","pct").replace("+","p").replace("/","_").replace(" ","_")
        for c in df_clean.columns
    ]
    spark.createDataFrame(df_clean) \
         .write.mode("overwrite") \
         .option("overwriteSchema","true") \
         .format("delta") \
         .saveAsTable(full_name)
    count = spark.table(full_name).count()
    print(f"✓ {full_name}: {count} rows saved")

def load_table(table_name, drop_strings=False):
    df_spark = spark.table(f"srm.{table_name}")
    if drop_strings:
        drop_cols = [c for c in STRING_COLS if c in df_spark.columns]
        if drop_cols:
            df_spark = df_spark.drop(*drop_cols)
    df = df_spark.toPandas()
    for col in df.columns:
        if col in ["ds","year_month"]:
            df[col] = pd.to_datetime(df[col])
    return df

print("=== Notebook 3: Indicator Scoring ===")

StatementMeta(, b5d36df3-a25a-43d4-a9e5-2d75ab4ff14e, 3, Finished, Available, Finished, False)

=== Notebook 3: Indicator Scoring ===


In [2]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: LOAD DATA
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 1: Loading data ===")

df_forecasts   = load_table("prophet_forecasts")
df_current     = load_table("prophet_current_values")
df_demand_curr = load_table("prophet_demand_current")
df_policy      = load_table("features_policy", drop_strings=True)
df_baseline    = load_table("prophet_baseline_summary")

print(f"Forecasts:      {df_forecasts.shape}")
print(f"Current values: {df_current.shape}")
print(f"Demand rolling: {df_demand_curr.shape}")
print(f"Policy:         {df_policy.shape}")
print(f"Baseline:       {df_baseline.shape}")

# ── Build baseline lookup dict ─────────────────────────────────────
# {(commodity, indicator): baseline_avg}
baseline_lookup = {}
for _, row in df_baseline.iterrows():
    key = (row["commodity"], row["indicator"])
    baseline_lookup[key] = float(row["baseline_avg"])

print(f"\nBaseline lookup entries: {len(baseline_lookup)}")
print("Sample baselines:")
for (commodity, indicator), avg in list(baseline_lookup.items())[:6]:
    print(f"  {commodity:<8} {indicator:<20} baseline = {avg:.3f}")

# ── Add demand into current and forecast ──────────────────────────
df_demand_for_current = df_demand_curr[
    df_demand_curr["period_type"]=="current"
][["ds","yhat","yhat_lower","yhat_upper","commodity"]].copy()
df_demand_for_current["indicator"]   = "demand_ratio"
df_demand_for_current["period_type"] = "current"
df_demand_for_current["ds"] = df_demand_for_current["ds"].astype("datetime64[us]")
df_current["ds"] = df_current["ds"].astype("datetime64[us]")

df_current = pd.concat(
    [df_current, df_demand_for_current], ignore_index=True
).drop_duplicates(
    subset=["ds","indicator","commodity"], keep="last"
).reset_index(drop=True)

df_demand_for_forecast = df_demand_curr[
    df_demand_curr["period_type"]=="forecast"
][["ds","yhat","yhat_lower","yhat_upper","commodity"]].copy()
df_demand_for_forecast["indicator"] = "demand_ratio"
df_demand_for_forecast["ds"] = df_demand_for_forecast["ds"].astype("datetime64[us]")
df_forecasts["ds"] = df_forecasts["ds"].astype("datetime64[us]")

df_forecasts = pd.concat(
    [df_forecasts, df_demand_for_forecast], ignore_index=True
).reset_index(drop=True)

# ── Combine current + forecast ─────────────────────────────────────
df_current["period_type"]   = "current"
df_forecasts["period_type"] = "forecast"

df_to_score = pd.concat([
    df_current[["ds","yhat","yhat_lower","yhat_upper",
                "indicator","commodity","period_type"]],
    df_forecasts[["ds","yhat","yhat_lower","yhat_upper",
                  "indicator","commodity","period_type"]]
], ignore_index=True)

df_to_score["ds"] = pd.to_datetime(df_to_score["ds"])
df_to_score = df_to_score.sort_values(
    ["commodity","indicator","ds"]
).reset_index(drop=True)

print(f"\nCombined scoring table: {df_to_score.shape}")
print(f"Indicators: {sorted(df_to_score['indicator'].unique())}")

StatementMeta(, b5d36df3-a25a-43d4-a9e5-2d75ab4ff14e, 4, Finished, Available, Finished, False)


=== Step 1: Loading data ===
Forecasts:      (48, 7)
Current values: (24, 9)
Demand rolling: (16, 10)
Policy:         (240, 21)
Baseline:       (20, 6)

Baseline lookup entries: 20
Sample baselines:
  Soybean  demand_ratio         baseline = 27590279.080
  Soybean  ppi_deviation        baseline = 101.512
  Soybean  ksa_deviation        baseline = 98.576
  Wheat    demand_ratio         baseline = 5968140.430
  Wheat    ppi_deviation        baseline = 100.620
  Wheat    ksa_deviation        baseline = 81.856

Combined scoring table: (84, 7)
Indicators: ['bdi_ratio', 'demand_ratio', 'ksa_deviation', 'policy_risk_weighted', 'ppi_deviation', 'stu_deviation']


In [3]:
# ── Step 1b: Load domestic reserves table ─────────────────────────
print("\n=== Step 1b: Loading domestic reserves ===")

df_reserves_raw = spark.table("srm.gld_domestic_reserves").toPandas()
df_reserves_raw.columns = [c.strip() for c in df_reserves_raw.columns]

# Rename columns for clarity
df_reserves_raw = df_reserves_raw.rename(columns={
    "Commodity":                      "commodity",
    "Target_Strategic_Reserve_KT_In_Months": "target_months",
    "Current_Stock_In_Months":        "current_months",
    "Reported_Date":                  "reported_date"
})

df_reserves_raw["reported_date"] = pd.to_datetime(df_reserves_raw["reported_date"])

print(f"Reserves table: {df_reserves_raw.shape}")
print(df_reserves_raw.to_string(index=False))

# Compute gap per commodity
df_reserves_raw["gap_months"] = (
    df_reserves_raw["target_months"] - df_reserves_raw["current_months"]
)

print(f"\nGap analysis (target - current, positive = below target = risk):")
print(df_reserves_raw[["commodity","target_months","current_months","gap_months"]].to_string(index=False))

StatementMeta(, b5d36df3-a25a-43d4-a9e5-2d75ab4ff14e, 5, Finished, Available, Finished, False)


=== Step 1b: Loading domestic reserves ===
Reserves table: (4, 4)
commodity  target_months  current_months reported_date
  Soybean            3.0             3.2    2026-06-24
    Wheat            4.0             4.0    2026-06-24
     Rice            7.0             8.8    2026-06-24
     Corn            3.0             3.4    2026-06-24

Gap analysis (target - current, positive = below target = risk):
commodity  target_months  current_months  gap_months
  Soybean            3.0             3.2        -0.2
    Wheat            4.0             4.0         0.0
     Rice            7.0             8.8        -1.8
     Corn            3.0             3.4        -0.4


In [4]:
# ══════════════════════════════════════════════════════════════════
# STEP 2: DEFINE SCORING FUNCTION 
# Converts Prophet output back to raw value first
# Then applies exact thresholds from image
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 2: Defining scoring function ===")

def interpolate(val, lo_thresh, hi_thresh, lo_score, hi_score):
    """Linear interpolation within a band."""
    if hi_thresh == lo_thresh:
        return float(lo_score)
    t = (val - lo_thresh) / (hi_thresh - lo_thresh)
    t = max(0.0, min(1.0, t))
    return round(lo_score + t * (hi_score - lo_score), 2)


def score_indicator(prophet_value, indicator, commodity,
                    baseline_lookup=baseline_lookup):
    """
    framework scoring.
    Step 1: Convert Prophet deviation/ratio → raw value
    Step 2: Apply thresholds from image

    Boss thresholds (Low/Watch/Warning/Emergency):
      STU:    25 / 20 / 15         raw %
      Demand: ratio 1.05/1.15/1.30 (mapped from  10/12/14/16)
      PPI:     3 /  0 / -3         raw deviation
      KSA:    15 / 25 / 50         raw %
      Policy: 10 / 40 / 70         raw score 0-100
      BDI:    30 / 60 / 90         normalised 0-100 score
    """
    if pd.isna(prophet_value):
        return np.nan

    # ── STEP 1: Convert Prophet output to raw value ────────────────

    if indicator == "stu_deviation":
        baseline = baseline_lookup.get((commodity, "stu_ratio"), 26.86)

        # % deviation from 5Y baseline (signed)
        # positive = stocks ABOVE average = good = low score
        # negative = stocks BELOW average = risk = high score
        pct_dev = (prophet_value / baseline) * 100

        STU_DEV_FLOOR_PCT = -25   # ⚠ assumption — most negative deviation% expected; adjust to your historical extremes

        if pct_dev >= 10:
            return 10.0
        elif pct_dev >= 0:
            return interpolate(pct_dev, 10, 0, 10, 40)
        elif pct_dev >= -10:
            return interpolate(pct_dev, 0, -10, 40, 70)
        else:
            return interpolate(max(pct_dev, STU_DEV_FLOOR_PCT), -10, STU_DEV_FLOOR_PCT, 70, 100)

    elif indicator == "ppi_deviation":
        # Prophet gave: deviation from baseline (ppi_base - baseline_avg)
        # ppi_base is already centred around 100
        # PPI values: 3, 0, -3, -6
        # = ppi_base - 100 (deviation from 100)
        # Our deviation = ppi_base - baseline_avg ≈ ppi_base - 100
        # since baseline_avg ≈ 100-101
        baseline = baseline_lookup.get((commodity, "ppi_deviation"), 100.0)
        raw = prophet_value + baseline - 100  # convert to deviation from 100

        # Boss thresholds on PPI deviation
        # Lower is worse
        if raw >= 3:    return 0.0
        elif raw >= 0:  return interpolate(raw, 3, 0, 0, 30)
        elif raw >= -3: return interpolate(raw, 0, -3, 30, 60)
        else:           return interpolate(max(raw, -10), -3, -10, 60, 80)

    elif indicator == "bdi_ratio":
        # Prophet gave: ratio = current_bdi / baseline_avg
        # Convert to 0-100 score:
        # score = (ratio - 0.5) / 1.5 × 100
        # ratio=0.5 → score=0   (very low freight)
        # ratio=1.0 → score=33  (at baseline)
        # ratio=1.3 → score=53  (30% above baseline)
        # ratio=1.6 → score=73  (60% above baseline)
        # ratio=2.0 → score=100 (double baseline)
        raw = min(100, max(0, (prophet_value - 0.5) / 1.5 * 100))

        # Boss thresholds on BDI 0-100 score
        # Higher is worse
        if raw < 30:    return 0.0
        elif raw < 60:  return interpolate(raw, 30, 60, 0, 30)
        elif raw < 90:  return interpolate(raw, 60, 90, 30, 60)
        else:           return interpolate(min(raw, 100), 90, 100, 60, 80)

    # NEW — thresholds per screenshot 2 ("Net import vs 5Y avg"), floor of 10
    elif indicator == "demand_ratio":
        pct = (prophet_value - 1.0) * 100   # ratio -> % deviation from baseline

        DEMAND_FLOOR_PCT = -25   # ⚠ assumption — % below which score flat-lines at 10; adjust if needed

        if pct < 0:
            return interpolate(max(pct, DEMAND_FLOOR_PCT), DEMAND_FLOOR_PCT, 0, 10, 40)
        elif pct < 10:
            return interpolate(pct, 0, 10, 40, 70)
        elif pct < 25:
            return interpolate(pct, 10, 25, 70, 100)
        else:
            return 100.0

    elif indicator == "ksa_deviation":
        # Prophet gave: deviation = current% - baseline%
        # Raw = deviation + baseline
        baseline = baseline_lookup.get((commodity, "ksa_top3_pct"),
                    baseline_lookup.get((commodity, "ksa_deviation"), 85.0))
        raw = prophet_value + baseline

        # Boss thresholds on raw KSA top3 %
        # Higher is worse
        # Note: all current values > 50% → will score Emergency
        # This is correct per boss's framework
        if raw < 15:    return 0.0
        elif raw < 25:  return interpolate(raw, 15, 25, 0, 30)
        elif raw < 50:  return interpolate(raw, 25, 50, 30, 60)
        else:           return interpolate(min(raw, 100), 50, 100, 60, 80)

    elif indicator == "policy_risk_weighted":
        # Already raw score 0-100 — no conversion needed
        raw = prophet_value

        # Boss thresholds
        # Higher is worse
        if raw < 10:    return 0.0
        elif raw < 40:  return interpolate(raw, 10, 40, 0, 30)
        elif raw < 70:  return interpolate(raw, 40, 70, 30, 60)
        else:           return interpolate(min(raw, 100), 70, 100, 60, 80)

    else:
        return np.nan


# ── Test scoring function ──────────────────────────────────────────
print("\nScoring verification :")
print(f"\n{'Indicator':<25} {'Prophet':>10} {'Raw':>10} {'Score':>8}  Expected")
print("-" * 72)

# NEW — full test block, all expected values verified against current score_indicator() logic
tests = [
    # STU: pct_dev = (prophet_value / baseline) * 100, baseline(Wheat)=26.86, floor -25%
    ("stu_deviation",  "Wheat",   2.686,  "pct_dev= 10.0% → expect  10.0  (Low threshold)"),
    ("stu_deviation",  "Wheat",   0.0,    "pct_dev=  0.0% → expect  40.0  (Watch threshold)"),
    ("stu_deviation",  "Wheat",  -1.343,  "pct_dev= -5.0% → expect  55.0  (mid Warning)"),
    ("stu_deviation",  "Wheat",  -2.686,  "pct_dev=-10.0% → expect  70.0  (Warning threshold)"),
    ("stu_deviation",  "Wheat",  -6.715,  "pct_dev=-25.0% → expect 100.0  (Emergency / floor)"),

    # PPI: raw = deviation + baseline - 100  (unchanged formula)
    ("ppi_deviation",  "Wheat",   2.38,  "raw=+3   → expect  0.0  (at Low threshold)"),
    ("ppi_deviation",  "Wheat",   0.38,  "raw= 0   → expect 30.0  (at Watch threshold)"),
    ("ppi_deviation",  "Wheat",  -2.62,  "raw=-3   → expect 60.0  (at Warning threshold)"),

    # BDI: score = (ratio - 0.5) / 1.5 × 100  (unchanged formula)
    ("bdi_ratio",      "Wheat",   0.95,  "score=30 → expect  0.0  (at Low threshold)"),
    ("bdi_ratio",      "Wheat",   1.40,  "score=60 → expect 30.0  (at Watch threshold)"),
    ("bdi_ratio",      "Wheat",   1.85,  "score=90 → expect 60.0  (at Warning threshold)"),
    ("bdi_ratio",      "Wheat",   1.142, "score=43 → expect 10.2  (current BDI forecast)"),

    # KSA: raw = deviation + baseline(83.9 for Wheat)  (unchanged formula)
    ("ksa_deviation",  "Wheat",  13.37,  "raw=97.3% → expect 74.0 (Emergency, >50%)"),
    ("ksa_deviation",  "Corn",   -0.07,  "raw=96.5% → expect 74.4 (Emergency, >50%)"),

    # Policy: raw score directly  (unchanged formula)
    ("policy_risk_weighted","Wheat",17.5, "raw=17.5 → expect  7.5  (Watch band, 10-40)"),
    ("policy_risk_weighted","Wheat",45.0, "raw=45.0 → expect 15.5  (Warning band, 40-70)"),

    # Demand: pct = (ratio - 1.0) * 100, floor -25%
    ("demand_ratio",   "Wheat",   0.75,   "pct=-25.0% → expect  10.0  (floor)"),
    ("demand_ratio",   "Wheat",   0.857,  "pct=-14.3% → expect  22.84 (below baseline)"),
    ("demand_ratio",   "Wheat",   1.0,    "pct=  0.0% → expect  40.0  (at baseline)"),
    ("demand_ratio",   "Wheat",   1.10,   "pct= 10.0% → expect  70.0  (Watch→Warning boundary)"),
    ("demand_ratio",   "Rice",    1.183,  "pct= 18.3% → expect  86.60 (Warning band)"),
    ("demand_ratio",   "Wheat",   1.25,   "pct= 25.0% → expect 100.0  (Emergency)"),
]

for indicator, commodity, prophet_val, expected in tests:
    score = score_indicator(prophet_val, indicator, commodity)

    # Compute raw for display
    if indicator == "stu_deviation":
        b = baseline_lookup.get((commodity,"stu_ratio"),26.86)
        raw_disp = (prophet_val / b) * 100
    elif indicator == "ppi_deviation":
        b = baseline_lookup.get((commodity,"ppi_deviation"),100.0)
        raw_disp = prophet_val + b - 100
    elif indicator == "bdi_ratio":
        raw_disp = min(100, max(0,(prophet_val-0.5)/1.5*100))
    elif indicator == "ksa_deviation":
        b = baseline_lookup.get((commodity,"ksa_top3_pct"),
            baseline_lookup.get((commodity,"ksa_deviation"),85.0))
        raw_disp = prophet_val + b
        raw_disp = min(raw_disp, 100.0)
    elif indicator == "demand_ratio":
        raw_disp = (prophet_val - 1.0) * 100
    else:
        raw_disp = prophet_val

    print(f"{indicator:<25} {prophet_val:>10.3f} {raw_disp:>10.3f} {score:>8.2f}  {expected}")

StatementMeta(, b5d36df3-a25a-43d4-a9e5-2d75ab4ff14e, 6, Finished, Available, Finished, False)


=== Step 2: Defining scoring function ===

Scoring verification :

Indicator                    Prophet        Raw    Score  Expected
------------------------------------------------------------------------
stu_deviation                  2.686      9.999    10.00  pct_dev= 10.0% → expect  10.0  (Low threshold)
stu_deviation                  0.000      0.000    40.00  pct_dev=  0.0% → expect  40.0  (Watch threshold)
stu_deviation                 -1.343     -5.000    55.00  pct_dev= -5.0% → expect  55.0  (mid Warning)
stu_deviation                 -2.686     -9.999    70.00  pct_dev=-10.0% → expect  70.0  (Warning threshold)
stu_deviation                 -6.715    -24.998   100.00  pct_dev=-25.0% → expect 100.0  (Emergency / floor)
ppi_deviation                  2.380      3.000     0.00  raw=+3   → expect  0.0  (at Low threshold)
ppi_deviation                  0.380      1.000    20.00  raw= 0   → expect 30.0  (at Watch threshold)
ppi_deviation                 -2.620     -2.000    50.0

In [5]:
# ══════════════════════════════════════════════════════════════════
# STEP 3: SCORE ALL INDICATORS
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 3: Scoring all indicators ===")

df_to_score["indicator_score"] = df_to_score.apply(
    lambda r: score_indicator(r["yhat"], r["indicator"], r["commodity"]), axis=1
)
df_to_score["indicator_score_lower"] = df_to_score.apply(
    lambda r: score_indicator(r["yhat_lower"], r["indicator"], r["commodity"]), axis=1
)
df_to_score["indicator_score_upper"] = df_to_score.apply(
    lambda r: score_indicator(r["yhat_upper"], r["indicator"], r["commodity"]), axis=1
)

print(f"Scored rows: {len(df_to_score)}")
print(f"NULL scores: {df_to_score['indicator_score'].isnull().sum()}")

print(f"\nScore distribution per indicator:")
print(df_to_score.groupby("indicator")["indicator_score"].describe().round(2))

StatementMeta(, b5d36df3-a25a-43d4-a9e5-2d75ab4ff14e, 7, Finished, Available, Finished, False)


=== Step 3: Scoring all indicators ===
Scored rows: 84
NULL scores: 0

Score distribution per indicator:
                      count   mean    std    min    25%    50%    75%    max
indicator                                                                   
bdi_ratio              16.0  19.73   9.31  12.77  14.10  15.48  21.10  35.19
demand_ratio           16.0  50.76  30.36  20.67  22.33  47.86  76.29  86.64
ksa_deviation          16.0  78.67   1.55  74.83  77.23  78.88  80.00  80.00
policy_risk_weighted    4.0   2.09   3.64   0.00   0.00   0.42   2.51   7.51
ppi_deviation          16.0  13.43  12.15   0.00   4.10  11.86  22.67  36.47
stu_deviation          16.0  46.78  25.24  17.30  35.20  40.02  52.05  91.27


In [6]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: POLICY SCORING 
# Current: actual policy_risk_score_weighted
# Forecast: 2% monthly decay
# thresholds: 10 / 40 / 70 / 100
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 4: Policy scoring ===")

POLICY_DECAY = 0.98
df_policy["year_month"] = pd.to_datetime(df_policy["year_month"])
policy_rows = []

for commodity in COMMODITIES:
    dc = df_policy[df_policy["commodity"]==commodity].copy()
    dc = dc.sort_values("year_month").reset_index(drop=True)

    current_row = dc[dc["year_month"] <= CURRENT_MONTH].tail(1)
    if current_row.empty:
        continue

    current_val   = float(current_row["policy_risk_score_weighted"].values[0])
    current_score = score_indicator(current_val, "policy_risk_weighted", commodity)

    policy_rows.append({
        "ds":              CURRENT_MONTH,
        "yhat":            current_val,
        "indicator_score": current_score,
        "indicator_score_lower": current_score,
        "indicator_score_upper": current_score,
        "indicator":       "policy_risk_weighted",
        "commodity":       commodity,
        "period_type":     "current"
    })

    print(f"\n{commodity}: policy={current_val:.2f} → score={current_score:.2f}")

    for i, month in enumerate(FORECAST_MONTHS):
        decay_factor   = POLICY_DECAY ** (i + 1)
        forecast_val   = round(current_val * decay_factor, 4)
        forecast_score = score_indicator(
            forecast_val, "policy_risk_weighted", commodity
        )
        policy_rows.append({
            "ds":              month,
            "yhat":            forecast_val,
            "indicator_score": forecast_score,
            "indicator_score_lower": forecast_score,
            "indicator_score_upper": forecast_score,
            "indicator":       "policy_risk_weighted",
            "commodity":       commodity,
            "period_type":     "forecast"
        })
        print(f"  {month.strftime('%b %Y')}: val={forecast_val:.2f} "
              f"score={forecast_score:.2f}")

df_policy_scores = pd.DataFrame(policy_rows)
df_policy_scores["ds"] = pd.to_datetime(df_policy_scores["ds"])

save_to_lakehouse(df_policy_scores, "prophet_policy_scores")

StatementMeta(, b5d36df3-a25a-43d4-a9e5-2d75ab4ff14e, 8, Finished, Available, Finished, False)


=== Step 4: Policy scoring ===

Wheat: policy=17.51 → score=7.51
  Jul 2026: val=17.16 score=7.16
  Aug 2026: val=16.81 score=6.81
  Sep 2026: val=16.48 score=6.48

Corn: policy=4.95 → score=0.00
  Jul 2026: val=4.85 score=0.00
  Aug 2026: val=4.75 score=0.00
  Sep 2026: val=4.66 score=0.00

Rice: policy=10.84 → score=0.84
  Jul 2026: val=10.63 score=0.63
  Aug 2026: val=10.41 score=0.41
  Sep 2026: val=10.21 score=0.21

Soybean: policy=5.83 → score=0.00
  Jul 2026: val=5.71 score=0.00
  Aug 2026: val=5.60 score=0.00
  Sep 2026: val=5.49 score=0.00
✓ srm.prophet_policy_scores: 16 rows saved


In [7]:
# ══════════════════════════════════════════════════════════════════
# STEP 4b: DOMESTIC RESERVES SCORING
# Rule-based — no Prophet, no baseline
# Uses reported M+1 data for all periods
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 4b: Domestic reserves scoring ===")

CURRENT_MONTH = pd.Timestamp("2026-06-01")
FORECAST_MONTHS = pd.to_datetime(
    ["2026-07-01","2026-08-01","2026-09-01"]
)

# NEW — thresholds per screenshot 3 (% of current vs target), floor of 10
def score_reserves_pct(pct_of_target):
    """
    pct_of_target = (current_months / target_months) * 100
      > 95%     -> 10   (at/above target)
      85% - 95% -> 30
      < 85%     -> 100  (severe shortfall)
    """
    if pd.isna(pct_of_target): return np.nan
    RESERVES_FLOOR_PCT = 50   # ⚠ assumption — % below which score caps at 100; adjust if needed

    if pct_of_target >= 95:
        return 10.0
    elif pct_of_target >= 85:
        return round(interpolate(pct_of_target, 95, 85, 10, 30), 2)
    else:
        return round(interpolate(max(pct_of_target, RESERVES_FLOOR_PCT), 85, RESERVES_FLOOR_PCT, 30, 100), 2)

# Verify scoring function
# NEW
print("\nReserves scoring verification:")
test_pcts = [(100.0,"at target→Low"), (95.0,"Low→Watch boundary"),
             (90.0,"mid Watch"), (85.0,"Watch→Warning boundary"),
             (70.0,"mid Warning"), (50.0,"Warning→Emergency floor"),
             (30.0,"below floor, capped")]
for pct, label in test_pcts:
    print(f"  pct_of_target={pct:>6.2f}% → score={score_reserves_pct(pct):>6.2f}  ({label})")

# Score all periods
reserves_rows = []

for commodity in COMMODITIES:
    res = df_reserves_raw[df_reserves_raw["commodity"]==commodity]
    if res.empty:
        print(f"  WARNING: No reserves data for {commodity}")
        continue

    # NEW
    gap            = float(res["gap_months"].values[0])
    target         = float(res["target_months"].values[0])
    current        = float(res["current_months"].values[0])
    pct_of_target  = round(current / target * 100, 2) if target else np.nan
    score          = score_reserves_pct(pct_of_target)

    print(f"\n{commodity}:")
    print(f"  Target: {target:.2f} months | Current: {current:.2f} months")
    print(f"  Gap:    {gap:.2f} months | Score: {score:.2f}")

    # Apply same score to all 4 periods
    # (M+2 and M+3 carry forward M+1 data)
    all_periods = [CURRENT_MONTH] + list(FORECAST_MONTHS)
    period_types = ["current","forecast","forecast","forecast"]

    for period, ptype in zip(all_periods, period_types):
        reserves_rows.append({
            "ds":              period,
            "indicator":       "domestic_reserves_gap",
            "commodity":       commodity,
            "period_type":     ptype,
            "yhat":            pct_of_target, 
            "gap_months":      gap,           # gap in months
            "target_months":   target,
            "current_months":  current,
            "indicator_score": score,
            "indicator_score_lower": score,
            "indicator_score_upper": score,
            "note": "M+1 data carried forward for M+2/M+3"
        })

df_reserves_scores = pd.DataFrame(reserves_rows)
df_reserves_scores["ds"] = pd.to_datetime(df_reserves_scores["ds"])

print(f"\nReserves scores: {df_reserves_scores.shape}")
save_to_lakehouse(df_reserves_scores, "prophet_reserves_scores")

StatementMeta(, b5d36df3-a25a-43d4-a9e5-2d75ab4ff14e, 9, Finished, Available, Finished, False)


=== Step 4b: Domestic reserves scoring ===

Reserves scoring verification:
  pct_of_target=100.00% → score= 10.00  (at target→Low)
  pct_of_target= 95.00% → score= 10.00  (Low→Watch boundary)
  pct_of_target= 90.00% → score= 20.00  (mid Watch)
  pct_of_target= 85.00% → score= 30.00  (Watch→Warning boundary)
  pct_of_target= 70.00% → score= 60.00  (mid Warning)
  pct_of_target= 50.00% → score=100.00  (Warning→Emergency floor)
  pct_of_target= 30.00% → score=100.00  (below floor, capped)

Wheat:
  Target: 4.00 months | Current: 4.00 months
  Gap:    0.00 months | Score: 10.00

Corn:
  Target: 3.00 months | Current: 3.40 months
  Gap:    -0.40 months | Score: 10.00

Rice:
  Target: 7.00 months | Current: 8.80 months
  Gap:    -1.80 months | Score: 10.00

Soybean:
  Target: 3.00 months | Current: 3.20 months
  Gap:    -0.20 months | Score: 10.00

Reserves scores: (16, 12)
✓ srm.prophet_reserves_scores: 16 rows saved


In [8]:
# ══════════════════════════════════════════════════════════════════
# STEP 5: COMBINE ALL INDICATOR SCORES (updated)
# Now includes domestic_reserves_gap alongside other 6 indicators
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 5: Combining all indicator scores ===")

df_prophet_scores = df_to_score[
    df_to_score["indicator"] != "policy_risk_weighted"
][[
    "ds","indicator","commodity","period_type",
    "yhat","indicator_score","indicator_score_lower","indicator_score_upper"
]].copy()
df_prophet_scores["ds"] = df_prophet_scores["ds"].astype("datetime64[us]")

df_policy_aligned = df_policy_scores[[
    "ds","indicator","commodity","period_type",
    "yhat","indicator_score","indicator_score_lower","indicator_score_upper"
]].copy()
df_policy_aligned["ds"] = df_policy_aligned["ds"].astype("datetime64[us]")

df_reserves_aligned = df_reserves_scores[[
    "ds","indicator","commodity","period_type",
    "yhat","indicator_score","indicator_score_lower","indicator_score_upper"
]].copy()
df_reserves_aligned["ds"] = df_reserves_aligned["ds"].astype("datetime64[us]")

df_all_scores = pd.concat([
    df_prophet_scores,
    df_policy_aligned,
    df_reserves_aligned    # ← NEW
], ignore_index=True)

df_all_scores["ds"] = pd.to_datetime(df_all_scores["ds"])
df_all_scores = df_all_scores.drop_duplicates(
    subset=["ds","indicator","commodity","period_type"],
    keep="last"
).sort_values(["commodity","indicator","ds"]).reset_index(drop=True)

print(f"All indicator scores: {df_all_scores.shape}")
print(f"Indicators: {sorted(df_all_scores['indicator'].unique())}")

save_to_lakehouse(df_all_scores, "prophet_indicator_scores")

StatementMeta(, b5d36df3-a25a-43d4-a9e5-2d75ab4ff14e, 10, Finished, Available, Finished, False)


=== Step 5: Combining all indicator scores ===
All indicator scores: (112, 8)
Indicators: ['bdi_ratio', 'demand_ratio', 'domestic_reserves_gap', 'ksa_deviation', 'policy_risk_weighted', 'ppi_deviation', 'stu_deviation']
✓ srm.prophet_indicator_scores: 112 rows saved


In [9]:
# ══════════════════════════════════════════════════════════════════
# STEP 6: SCORE REVIEW 
# Shows: raw converted value + score side by side
# ══════════════════════════════════════════════════════════════════

print("\n=== Step 6: Score review ===")

all_periods    = pd.to_datetime([
    "2026-06-01","2026-07-01","2026-08-01","2026-09-01"
])
period_labels  = ["Jun 2026","Jul 2026","Aug 2026","Sep 2026"]

# NEW
REFERENCE_THRESHOLDS = {
    "stu_deviation":         {"Low":10, "Watch":0, "Warning":-10, "Emergency":-25},
    "demand_ratio":          {"Low":0, "Watch":10, "Warning":25, "Emergency":">=25"},
    "ppi_deviation":         {"Low":3, "Watch":0, "Warning":-3, "Emergency":-6},
    "ksa_deviation":         {"Low":15, "Watch":25, "Warning":50, "Emergency":">=50"},
    "policy_risk_weighted":  {"Low":10, "Watch":40, "Warning":70, "Emergency":100},
    "bdi_ratio":             {"Low":30, "Watch":60, "Warning":90, "Emergency":">=90"},
    "domestic_reserves_gap": {"Low":100, "Watch":95, "Warning":85, "Emergency":50},   # NEW
}

for commodity in COMMODITIES:
    print(f"\n{'═'*105}")
    print(f"  {commodity}")
    print(f"{'═'*105}")
    print(f"{'Indicator':<28} {'Wt':>4}  "
          f"{'Jun 2026':>18} {'Jul 2026':>18} {'Aug 2026':>18} {'Sep 2026':>18}")
    print(f"{'-'*105}")

    for indicator in INDICATORS_ORDERED:
        weight = WEIGHTS.get(indicator, 0)
        print(f"{INDICATOR_LABELS[indicator]:<28} {weight*100:>3.0f}%  ", end="")

        for period in all_periods:
            row = df_all_scores[
                (df_all_scores["commodity"]==commodity) &
                (df_all_scores["indicator"]==indicator) &
                (df_all_scores["ds"]==period)
            ]
            if row.empty:
                print(f"{'N/A':>18}", end="")
            else:
                score = row["indicator_score"].values[0]
                raw   = row["yhat"].values[0]
                # Show converted raw value
                if indicator == "stu_deviation":
                    b = baseline_lookup.get((commodity,"stu_ratio"),26.86)
                    disp = (raw / b) * 100
                elif indicator == "ppi_deviation":
                    b = baseline_lookup.get((commodity,"ppi_deviation"),100.0)
                    disp = raw + b - 100
                elif indicator == "bdi_ratio":
                    disp = min(100, max(0,(raw-0.5)/1.5*100))
                elif indicator == "ksa_deviation":
                    b = baseline_lookup.get(
                        (commodity,"ksa_top3_pct"),
                        baseline_lookup.get((commodity,"ksa_deviation"),85.0)
                    )
                    disp = raw + b
                elif indicator == "demand_ratio":
                    disp = raw   # raw here is already prophet_value (the ratio); convert:
                    disp = (raw - 1.0) * 100
                else:
                    disp = raw
                print(f"{score:>8.1f}({disp:>7.1f})", end="")
        print()

    print(f"\n  Format: score(raw_value)")
    print(f"  Boss thresholds:")
    for indicator in INDICATORS_ORDERED:
        t = REFERENCE_THRESHOLDS[indicator]
        print(f"    {INDICATOR_LABELS[indicator]:<28}: "
              f"Low={t['Low']}  Watch={t['Watch']}  "
              f"Warning={t['Warning']}  Emergency={t['Emergency']}")

print(f"\n=== NOTEBOOK 3 COMPLETE ===")
print(f"""
Tables saved:
  srm.prophet_indicator_scores
  srm.prophet_policy_scores

Next: Notebook 4 — Final Risk Score + Risk Band
""")

StatementMeta(, b5d36df3-a25a-43d4-a9e5-2d75ab4ff14e, 11, Finished, Available, Finished, False)


=== Step 6: Score review ===

═════════════════════════════════════════════════════════════════════════════════════════════════════════
  Wheat
═════════════════════════════════════════════════════════════════════════════════════════════════════════
Indicator                      Wt            Jun 2026           Jul 2026           Aug 2026           Sep 2026
---------------------------------------------------------------------------------------------------------
Global Stock-to-Use Ratio     18%      43.3(   -1.1)    40.9(   -0.3)    39.4(    0.2)    37.9(    0.7)
Import Demand Pressure        12%      22.9(  -14.3)    22.9(  -14.3)    22.9(  -14.3)    22.9(  -14.3)
Production Potential Index    11%      36.5(   -0.6)     7.0(    2.3)     6.3(    2.4)     5.5(    2.5)
KSA Import Concentration      13%      74.8(   87.1)    79.9(   99.6)    80.0(  100.1)    80.0(  100.6)
Export Restriction Status     18%       7.5(   17.5)     7.2(   17.2)     6.8(   16.8)     6.5(   16.5)
BDI Freight 